In [ ]:
from typing import Any

from rockyclickup import wrapper
from rockyclickup.database_interface import get_custom_field_by_name

import datetime as dt

from rockyclickup.utils import convert_datetime, response_to_model


In [ ]:
rcu = wrapper.Session()

In [ ]:
GATEWAY_TEST_LIST_ID = 901114339591

In [ ]:

class DataFileCreateError(RuntimeError):
    """A DataFile create call came back without a usable task id."""
    
class FieldResolutionError(RuntimeError):
    """A DataFile field name has no matching ClickUp custom field."""



In [ ]:
_fields = {}

def _field(name: str):
    """Resolve (and cache) the ClickUp custom field behind a DataFile attribute."""
    if name not in _fields:
        found = get_custom_field_by_name(name)
        if found is None:
            raise FieldResolutionError(f"no ClickUp custom field named {name!r}")

        _fields[name] = found

    return _fields[name]


def field_id(name: str) -> str:
    return _field(name).field_id


def _field_value(value: Any) -> Any:
    return convert_datetime(value) if isinstance(value, dt.datetime) else value


In [ ]:

def create_datafile(name: str, fields: dict[str, Any]) -> str:
    """Create a DataFile task and return its new task id.

    Args:
        name:   the task name (top-level, not a custom field).
        fields: DataFile attribute name -> value, written as custom fields.
    """
    custom_fields = []
    for field_name, value in fields.items():
        if value is None:
            continue

        custom_fields.append({"id": field_id(field_name), "value": _field_value(value)})

    payload = {"name": name, "custom_fields": custom_fields}
    url = f"{rcu.base}/list/{GATEWAY_TEST_LIST_ID}/task"
    response = rcu.post(endpoint=url, payload=payload)

    task_id = response.get("id")
    if not task_id:
        raise DataFileCreateError(f"create returned no task id for {name!r}: {response}")

    print(f"created DataFile {task_id} for {name!r}")
    return task_id

In [ ]:
task_id = create_datafile(
    name="TEST_FILENAME",
    fields={
        "ftp_user": "TEST_USER",
        "ftp_filename": "TEST_FILENAME",
        "ftp_directory": "TEST_DIRECTORY",
        "file_date": dt.datetime.now(),
        "received": dt.datetime.now(),
        "file_category": "433537c3-29b7-494d-afae-0d9603649c30",
    },
)